# ENTSO-E Data Preprocessing

**Time Index**

* Convert all time indices to UTC before any other step.
* Resample all datasets to a common hourly UTC grid (mean of 15-min values), which also turns missing timestamps into missing values.

**Missing values**

* Gaps of up to 3 hours – linear interpolation: the missing 30 November 00:00 price timestamp in all non-PL zones, DST-related gaps in DE_LU/FR and isolated gaps in FR `generation_forecast`, LT `load_forecast` and LT `generation_wind_solar_forecast`.
* Longer gaps in weather-driven series (`generation_wind_solar_forecast`) – copy the same hours from the previous available day (D-1): FR `Solar`/`Wind Onshore` (2026-01-02), HU `Wind Onshore` (2026-02-16, 15:30–23:45), the whole-day gap in FR (2023-04-18) and the five gaps in SE_4 (for the 2-day gap use the last available day before it).
* Longer gaps in all other series (demand- and schedule-driven) – copy the same hours from one week earlier (D-7): SE_4 `load_forecast` (2025-12-04), `generation_forecast` in FR (2025-03-06), SK (2023-12-08) and SE_4 (2025-12-02).
* D-1 is used for weather-driven series because weather persists from one day to the next, and D-7 for demand- and schedule-driven series because they follow a weekly (weekday/weekend) cycle. If the source day is missing as well, the next earlier day is used. The 3-hour limit keeps interpolation within a small part of the daily cycle (solar, load); the gaps found are either up to 1 hour or at least 8.5 hours long, so no case lies close to the limit.
* Reservoir data (AT, NO, SE): forward fill only, including the 14-day step; the series are extended with the last value to the end of the other datasets.
* PL: no gaps to fill, only the UTC conversion and hourly resampling change the data.

In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.preprocessing import preprocess_entsoe
from src.utils.file_io import inspect_entsoe_files, save_entsoe_data, load_entsoe_data

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_entsoe = load_entsoe_data(
    DATA_DIR / "clean" / "entsoe",
    date_range,
)

In [4]:
data_entsoe = preprocess_entsoe(data_entsoe)

## 1. Index Integrity – Confirmation

All datasets in all bidding zones now share a UTC index with a single 1-hour step, without duplicated timestamps and without Summer/Winter time repeats, so the gaps in the time index are gone. The datasets differ only in the period they cover.

In [5]:
for code, datasets in data_entsoe.items():
    print(f"===== {code} {"=" * 20}")
    for name, data in datasets.items():
        print_index_summary(data=data, name=name)

===== AT ====================

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33238

--- [water_reservoirs_hydro_storage] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33311

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33239

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33238

--- [generation_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33238
===== CZ ====================

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps

## 2. Missing Values – Confirmation

No missing values remain in any dataset: all gaps were filled according to the strategies above.

In [6]:
for code, datasets in data_entsoe.items():
    print(f"===== {code} {"=" * 20}")
    for name, data in datasets.items():
        print_missing_values(data=data, name=name)

===== AT ====================

--- [load_forecast] Missing values ---
No missing values.

--- [water_reservoirs_hydro_storage] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== CZ ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== DE_LU ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== FR ====================

--- [load_fore

## 3. Conclusions

The preprocessing decisions from the data quality audit have been applied to all bidding zones, and the checks above confirm the result. The preprocessed datasets are saved to `data/preprocessed/entsoe/` and verified by reading the files back.

## 4. Saving Preprocessed Data

In [ ]:
save_entsoe_data(
    data_entsoe,
    DATA_DIR / "preprocessed" / "entsoe",
    date_range,
)

INFO:src.utils.file_io:Saved preprocessed/entsoe/AT/2022-12-01_2026-09-16/load_forecast.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/AT/2022-12-01_2026-09-16/water_reservoirs_hydro_storage.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/AT/2022-12-01_2026-09-16/day_ahead_prices.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/AT/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/AT/2022-12-01_2026-09-16/generation_forecast.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/CZ/2022-12-01_2026-09-16/load_forecast.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/CZ/2022-12-01_2026-09-16/day_ahead_prices.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/CZ/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/CZ/2022-12-01_2026-09-16/generation_forecast.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/DE_LU/2022-12-01_2

In [ ]:
inspect_entsoe_files(
    DATA_DIR / "preprocessed" / "entsoe",
    date_range,
)


--- AT (5 file/s) ------------------------------
  [FILE] data/preprocessed/entsoe/AT/2022-12-01_2026-09-16/load_forecast.parquet | shape: (33239, 1) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 21:00:00+00:00
  [FILE] data/preprocessed/entsoe/AT/2022-12-01_2026-09-16/water_reservoirs_hydro_storage.parquet | shape: (33312, 1) | idx: 2022-11-27 23:00:00+00:00-2026-09-15 22:00:00+00:00
  [FILE] data/preprocessed/entsoe/AT/2022-12-01_2026-09-16/day_ahead_prices.parquet | shape: (33240, 1) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 22:00:00+00:00
  [FILE] data/preprocessed/entsoe/AT/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet | shape: (33239, 2) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 21:00:00+00:00
  [FILE] data/preprocessed/entsoe/AT/2022-12-01_2026-09-16/generation_forecast.parquet | shape: (33239, 2) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 21:00:00+00:00

--- CZ (4 file/s) ------------------------------
  [FILE] data/preprocessed/entsoe/CZ/2022-12-01_2026-09-1